In [13]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

output_dir = "filtering_figures"
os.makedirs(output_dir, exist_ok=True)

In [10]:
# Creates boxplots comparing distribution of regions before and after filtering

# Load both before and after filtering for direct comparison
bed_raw = pd.read_csv("all_peaks_200bp.bed", sep="\t", header=None, usecols=[0, 1, 2], names=["chrom", "start", "end"])
bed_filt = pd.read_csv("filtered_peaks.bed", sep="\t", header=None, usecols=[0, 1, 2], names=["chrom", "start", "end"])

# Filter chromosomes
main_chroms = ["chr2L", "chr2R", "chr3L", "chr3R", "chr4", "chrX", "chrY"]
bed_raw = bed_raw[bed_raw["chrom"].isin(main_chroms)]
bed_filt = bed_filt[bed_filt["chrom"].isin(main_chroms)]

# Label datasets
bed_raw["status"] = "Before Filtering"
bed_filt["status"] = "After Filtering"
combined = pd.concat([bed_raw, bed_filt])
combined["chrom"] = pd.Categorical(combined["chrom"], categories=main_chroms, ordered=True)

# Plot with seaborn
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 6))
sns.boxplot(data=combined, x="chrom", y="start", hue="status", showfliers=True)
plt.xticks(rotation=45, ha="right")
plt.title("Region Start Position Distributions per Chromosome\nBefore vs After Filtering")
plt.ylabel("Start position (bp)")
plt.xlabel("Chromosome")
plt.tight_layout()
plt.savefig("region_distribution_comparison.png", dpi=300)
plt.close()

In [11]:
# Region distribution plot for each chromosome
# Run for before and after blacklist filtering to visualize the change in regions.

# Inputs
bed_path = "all_peaks_200bp.bed"  # Change this between unfiltered / filtered BED files
fai_path = "data/dm6.fa.fai"


# Determine filtering status from filename
if "filtered" in bed_path.lower():
    status = "After Filtering"
else:
    status = "Before Filtering"

# Load BED
bed = pd.read_csv(bed_path, sep="\t", header=None, usecols=[0, 1, 2], names=["chrom", "start", "end"])

# Main chroms
main_chroms = ["chr2L", "chr2R", "chr3L", "chr3R", "chr4", "chrX", "chrY"]
bed = bed[bed["chrom"].isin(main_chroms)]

# Plot: density per chromosome
plt.figure(figsize=(12, 6))
for chrom in main_chroms:
    starts = bed[bed["chrom"] == chrom]["start"]
    if len(starts) > 0:
        sns.kdeplot(starts, label=chrom, linewidth=1.2)


plt.suptitle(f"Region Start Density per Chromosome ({status})", fontsize=14, fontweight='bold')
plt.xlabel("Genomic position")
plt.ylabel("Density")
plt.legend()
plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust for suptitle

# Save figure with dynamic filename
filename = f"region_start_density_per_chromosome_{status.lower().replace(' ', '_')}.png"
plt.savefig(os.path.join(output_dir, filename), dpi=300)
plt.close()


In [14]:
# Region distribution figures for each chromosome before and after the blacklist filtering of the bed file of peaks divided into 200 bp windows.

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# === Inputs ===
bed_raw_path = "all_peaks_200bp.bed"    
bed_filt_path = "filtered_peaks.bed"    

# Load both BEDs
bed_raw = pd.read_csv(bed_raw_path, sep="\t", header=None, usecols=[0, 1], names=["chrom", "start"])
bed_filt = pd.read_csv(bed_filt_path, sep="\t", header=None, usecols=[0, 1], names=["chrom", "start"])

# Chromosomes to compare (standard Drosophila)
chroms = ["chr2L", "chr2R", "chr3L", "chr3R", "chr4", "chrX", "chrY"]

# Loop and plot
for chrom in chroms:
    starts_raw = bed_raw[bed_raw["chrom"] == chrom]["start"]
    starts_filt = bed_filt[bed_filt["chrom"] == chrom]["start"]

    if len(starts_raw) < 5 or len(starts_filt) < 5:
        continue  # Skip empty or tiny chroms

    plt.figure(figsize=(8, 5))
    sns.kdeplot(starts_raw, label="Original", color="gray", linestyle="--", bw_adjust=1)
    sns.kdeplot(starts_filt, label="Filtered", color="blue", bw_adjust=1)
    plt.title(f"Region Start Density on {chrom}")
    plt.xlabel("Genomic Start Position")
    plt.ylabel("Density")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"compare_density_{chrom}.png"), dpi=300)
    plt.close()




In [1]:
# Generates scatter plots for ATAC-seq data
# Compares signals of orer to each mutant (zldrnai, cic6, gd7) for each time point (1to15, 15to2, 2to25, 25to3)

import os
import matplotlib.pyplot as plt
import h5py
import numpy as np
import seaborn as sns

# Define timepoints and mutant names
timepoints = ["1to15", "15to2", "2to25", "25to3"]
mutants = ["zldrnai", "gd7", "cic6"]

# Function to get experiment names
def get_exp_name(mutant, timepoint):
    return f"{mutant}_{timepoint}_atac"

# Function for Quantile Normalization 
# This function normalizes the columns of a matrix to have the same distribution

def quantile_normalize(matrix):
    """
    Performs quantile normalization on the columns of a matrix based on the provided formula.
    """
    x = matrix.copy() # Use a copy to avoid modifying original data
    # Sort each column and get the row-wise mean to create the rank-ordered mean vector
    ranked_mean = np.mean(np.sort(x, axis=0), axis=1)
    # Get the rank of each element in each column
    ranks = np.argsort(np.argsort(x, axis=0), axis=0)
    # Map the ranked means back to the original element positions
    return ranked_mean[ranks]

# File path
h5_path = "combined_all_signals_after_filtering.h5"
# Load signal_matrix and experiment_types from H5 file
with h5py.File(h5_path, "r") as f:
    signal_matrix = f["signal_matrix"][:]  # shape: (regions, experiments)
    experiment_types = [s.decode("utf-8") for s in f["experiment_types"][:]]  # list of strings

normalized_signal_matrix = quantile_normalize(signal_matrix)

# Create a mapping: experiment name -> column index
name_to_index = {name: i for i, name in enumerate(experiment_types)}

# Output folder
output_dir = "region_distribution_figures"
# Directory for original data plots
scatter_dir = os.path.join(output_dir, "scatterplots_atac")
os.makedirs(scatter_dir, exist_ok=True)
# Directory for quantile normalized data plots
scatter_dir_qnorm = os.path.join(output_dir, "scatterplots_atac_qnorm")
os.makedirs(scatter_dir_qnorm, exist_ok=True)


# Generate scatter plots
for tp in timepoints:
    orer_name = get_exp_name("orer", tp)

    if orer_name not in name_to_index:
        print(f"Skipping {tp} orer: missing data")
        continue

    x_idx = name_to_index[orer_name]
    # Get original and quantile normalized data for the x-axis
    x_signal = np.log1p(signal_matrix[:, x_idx])
    x_signal_qnorm = np.log1p(normalized_signal_matrix[:, x_idx])

    for mutant in mutants:
        mutant_name = get_exp_name(mutant, tp)

        if mutant_name not in name_to_index:
            print(f"Skipping {tp} {mutant}: missing data")
            continue

        y_idx = name_to_index[mutant_name]
        # Get original and quantile normalized data for the y-axis
        y_signal = np.log1p(signal_matrix[:, y_idx])
        y_signal_qnorm = np.log1p(normalized_signal_matrix[:, y_idx])

        # --- PLOT 1: Original Data (Modified) ---
        plt.figure(figsize=(6, 6))
        plt.scatter(x_signal, y_signal, alpha=0.2, s=5) # Reduced alpha for clarity with contours

        # --- START: Added Diagonal and Contour ---
        sns.kdeplot(x=x_signal, y=y_signal, levels=5, colors='k', linewidths=0.7)
        lims = [min(plt.xlim()[0], plt.ylim()[0]), max(plt.xlim()[1], plt.ylim()[1])]
        plt.plot(lims, lims, 'r--', alpha=0.75, zorder=5, label='y=x') 
        
        plt.xlim(lims)
        plt.ylim(lims)
        plt.xlabel(f"log(1 + {orer_name} signal)") 
        plt.ylabel(f"log(1 + {mutant_name} signal)")
        plt.title(f"ATAC: {orer_name} vs {mutant_name}")
        plt.legend()
        plt.tight_layout()
        fname = f"scatter_{orer_name}_vs_{mutant_name}.png"
        plt.savefig(os.path.join(scatter_dir, fname), dpi=300)
        plt.close()

        # --- PLOT 2: Quantile Normalized Data ---
        plt.figure(figsize=(6, 6))
        plt.scatter(x_signal_qnorm, y_signal_qnorm, alpha=0.2, s=5)
        sns.kdeplot(x=x_signal_qnorm, y=y_signal_qnorm, levels=5, colors='k', linewidths=0.7)
        lims_qnorm = [min(plt.xlim()[0], plt.ylim()[0]), max(plt.xlim()[1], plt.ylim()[1])]

        # --- MODIFIED: Increased zorder to bring line to the front ---
        plt.plot(lims_qnorm, lims_qnorm, 'r--', alpha=0.75, zorder=5, label='y=x')

        plt.xlim(lims_qnorm)
        plt.ylim(lims_qnorm)
        plt.xlabel(f"log(1 + {orer_name} signal)")
        plt.ylabel(f"log(1 + {mutant_name} signal)")
        plt.title(f"ATAC: {orer_name} vs {mutant_name} (Quantile Normalized)")
        plt.legend()
        plt.tight_layout()
        fname_qnorm = f"scatter_qnorm_{orer_name}_vs_{mutant_name}.png"
        plt.savefig(os.path.join(scatter_dir_qnorm, fname_qnorm), dpi=300)
        plt.close()

print(f"All original scatter plots saved in: {scatter_dir}")
print(f"All quantile normalized scatter plots saved in: {scatter_dir_qnorm}")

Skipping 1to15 cic6: missing data
Skipping 15to2 cic6: missing data
All original scatter plots saved in: region_distribution_figures/scatterplots_atac
All quantile normalized scatter plots saved in: region_distribution_figures/scatterplots_atac_qnorm
